In [1]:
import json
import random
from datetime import datetime, timedelta
import uuid
import os

# ── Configuration ──────────────────────────────────────────────────────────────
NUM_LOGS        = 10000
ANOMALY_RATIO   = 0.05          # 5 % anomalies
OUTPUT_FILE     = "data/synthetic_logs.json"

SERVICES = [
    "user-service",
    "payment-service",
    "api-gateway",
    "k8s-kubelet",
    "docker-daemon",
]

# ── Helpers ────────────────────────────────────────────────────────────────────
JAVA_CLASSES = [
    "com.devinsight.auth.JwtFilter",
    "com.devinsight.user.UserService",
    "com.devinsight.payment.PaymentController",
    "com.devinsight.gateway.RouteFilter",
    "org.springframework.security.web.FilterChainProxy",
    "org.hibernate.engine.jdbc.spi.SqlExceptionHelper",
    "com.devinsight.common.DatabaseConnectionPool",
    "org.springframework.web.servlet.DispatcherServlet",
]

HTTP_METHODS  = ["GET", "POST", "PUT", "DELETE", "PATCH"]
HTTP_AGENTS   = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "PostmanRuntime/7.28.4",
    "curl/7.68.0",
    "python-requests/2.28.0",
    "Java/17.0.2",
    "okhttp/4.9.3",
]
DB_CLUSTERS   = ["cluster-01", "cluster-02", "replica-01", "read-replica-03"]
NAMESPACES    = ["default", "production", "staging", "monitoring"]
WORKER_NODES  = ["worker-node-1", "worker-node-2", "worker-node-3", "master-node-1"]
IMAGES        = [
    "devinsight/user-service:v1.2",
    "devinsight/payment-service:v2.0",
    "devinsight/api-gateway:v1.5",
    "openjdk:17-alpine",
    "postgres:14",
    "nginx:1.21",
]
CURRENCIES    = ["USD", "EUR", "INR", "GBP", "JPY"]
GATEWAYS      = ["stripe", "razorpay", "paypal", "braintree"]
VOLUMES       = ["/var/data/postgres", "/mnt/logs", "/etc/secrets", "/app/uploads"]

def rip(n=4):
    """Random IP last octet."""
    return random.randint(2, 254)

def rtx():
    return f"tx-{uuid.uuid4().hex[:8].upper()}"

def rpod(svc="service"):
    return f"{svc}-{random.randint(1,9)}{uuid.uuid4().hex[:5]}"

def rclass():
    c = random.choice(JAVA_CLASSES)
    line = random.randint(10, 250)
    return f"{c}.doFilter({c.split('.')[-1]}.java:{line})"

def rnode():
    return random.choice(WORKER_NODES)

def rns():
    return random.choice(NAMESPACES)

def rimg():
    return random.choice(IMAGES)

def rvol():
    return random.choice(VOLUMES)

def rgw():
    return random.choice(GATEWAYS)

def rcur():
    return random.choice(CURRENCIES)

def stacktrace(depth=3):
    lines = [f"java.lang.RuntimeException: Unexpected error"]
    for _ in range(depth):
        lines.append(f"\tat {rclass()}")
    lines.append(f"\tat java.lang.Thread.run(Thread.java:834)")
    return " ".join(lines)

# ── NORMAL TEMPLATES ───────────────────────────────────────────────────────────
NORMAL_TEMPLATES = {

    "user-service": [
        # Auth & session
        ("INFO",  "User login successful for username: user_{uid}"),
        ("INFO",  "JWT token validated successfully for user_{uid} [exp_in=3600s]"),
        ("INFO",  "Session created: session_id=sess-{hex} user=user_{uid}"),
        ("INFO",  "Token refreshed for user_{uid} new_exp=3600s"),
        ("INFO",  "Password reset link sent to user_{uid}@example.com"),
        ("INFO",  "User user_{uid} logged out. Session invalidated."),
        ("INFO",  "Account user_{uid} unlocked by admin after cooldown"),
        ("INFO",  "OAuth2 token exchange successful provider=google user={uid}"),
        # DB / pool
        ("DEBUG", "HikariPool-1 - Connection acquired in 4ms. pool_size=10 active=2"),
        ("DEBUG", "HikariPool-1 - Start completed. pool_size=10"),
        ("INFO",  "Fetching user profile for id: {uid}"),
        ("DEBUG", "SELECT * FROM users WHERE id={uid} executed in 8ms"),
        # Spring startup
        ("INFO",  "Started UserServiceApplication in 3.241s (JVM running for 3.9)"),
        ("INFO",  "Tomcat started on port(s): 8081 (http)"),
        ("INFO",  "Mapped to GET /api/users/{uid}"),
        # Health
        ("DEBUG", "Health check responded UP [db=UP, disk=UP, ping=UP]"),
        ("INFO",  "Scheduled task: purging expired sessions — removed 12 records"),
    ],

    "payment-service": [
        # Successful flows
        ("INFO",  "Processing payment of {amount} {currency} for transaction {tx}"),
        ("INFO",  "Payment {tx} authorized successfully via {gateway} in 340ms"),
        ("INFO",  "Idempotency key {hex} already processed — returning cached response"),
        ("INFO",  "Refund initiated for {tx} amount={amount} {currency} reason=customer_request"),
        ("INFO",  "Refund {tx} completed successfully"),
        ("INFO",  "Currency conversion: {amount} USD → {conv_amount} {currency} rate=0.82"),
        ("INFO",  "Fraud score for {tx}: 0.04 (threshold=0.8) — ALLOWED"),
        ("INFO",  "Circuit breaker [{gateway}] state=CLOSED success_rate=99.2%"),
        ("INFO",  "Rate limit check for merchant_{uid}: 48/50 requests used this minute"),
        # DB / JPA
        ("DEBUG", "Transaction {tx} persisted to DB in 12ms"),
        ("DEBUG", "SELECT * FROM payments WHERE tx_id='{tx}' executed in 6ms"),
        ("INFO",  "Connecting to payment gateway {gateway}: OK latency=45ms"),
        # Spring
        ("INFO",  "Started PaymentServiceApplication in 4.1s"),
        ("DEBUG", "Health check responded UP [gateway={gateway}, db=UP]"),
    ],

    "api-gateway": [
        # Nginx-style access logs — 2xx
        ("INFO", "192.168.1.{ip} - - [{date}] \"GET /api/v1/users HTTP/1.1\" 200 {bytes} \"-\" \"{agent}\""),
        ("INFO", "192.168.1.{ip} - - [{date}] \"POST /api/v1/login HTTP/1.1\" 200 {bytes} \"-\" \"{agent}\""),
        ("INFO", "192.168.1.{ip} - - [{date}] \"PUT /api/v1/users/{uid} HTTP/1.1\" 200 {bytes} \"-\" \"{agent}\""),
        ("INFO", "192.168.1.{ip} - - [{date}] \"GET /api/v1/payments/{tx} HTTP/1.1\" 200 {bytes} \"-\" \"{agent}\""),
        ("INFO", "192.168.1.{ip} - - [{date}] \"DELETE /api/v1/sessions HTTP/1.1\" 204 0 \"-\" \"{agent}\""),
        # Gateway internals
        ("INFO",  "Route matched: /api/v1/users/** → user-service:8081 [latency=12ms]"),
        ("INFO",  "Route matched: /api/v1/payments/** → payment-service:8082 [latency=18ms]"),
        ("DEBUG", "JWT validated for user_{uid} role=ROLE_DEVOPS"),
        ("INFO",  "Health probe: user-service UP, payment-service UP"),
        ("INFO",  "Rate limit check 192.168.1.{ip}: 120/500 requests this window"),
        ("INFO",  "SSL certificate valid. expires_in=87 days"),
        ("INFO",  "CORS preflight accepted for origin=http://localhost:5173"),
    ],

    "k8s-kubelet": [
        # Pod lifecycle — normal
        ("INFO",  "Successfully assigned {ns}/user-service-{pod} to {node}"),
        ("INFO",  "Container image \"{img}\" already present on machine"),
        ("INFO",  "Created container user-service"),
        ("INFO",  "Started container user-service"),
        ("INFO",  "Pod {ns}/payment-service-{pod} status changed: Pending → Running"),
        ("INFO",  "Liveness probe succeeded for {ns}/user-service-{pod} http_get=/actuator/health"),
        ("INFO",  "Readiness probe succeeded for {ns}/payment-service-{pod}"),
        # HPA / scaling
        ("INFO",  "HorizontalPodAutoscaler: scaling user-service from 2 → 3 replicas (CPU=72%)"),
        ("INFO",  "HorizontalPodAutoscaler: scaling down payment-service 3 → 2 replicas (CPU=18%)"),
        # PV
        ("INFO",  "Successfully mounted PersistentVolume {vol} on {ns}/postgres-0"),
        # GC / resource
        ("DEBUG", "GarbageCollector: collected 240 MB in 180ms (G1GC pause)"),
        ("INFO",  "Node {node} condition: Ready=True MemoryPressure=False DiskPressure=False"),
        ("DEBUG", "Syncing pod {ns}/user-service-{pod}"),
    ],

    "docker-daemon": [
        ("INFO",  "level=info msg=\"Container {cid} started successfully\""),
        ("INFO",  "level=info msg=\"Network devinsight_default created driver=bridge\""),
        ("INFO",  "level=info msg=\"Container {cid} health check passed\""),
        ("INFO",  "level=info msg=\"Image {img} pulled successfully size=245MB\""),
        ("INFO",  "level=info msg=\"Volume {vol} created successfully\""),
        ("INFO",  "level=info msg=\"Container {cid} gracefully stopped (exit 0)\""),
        ("DEBUG", "level=debug msg=\"Layer sha256:{hex}...{hex} already cached — skipping pull\""),
        ("INFO",  "level=info msg=\"Pruning unused images freed=1.2GB\""),
        ("INFO",  "level=info msg=\"Docker daemon listening on /var/run/docker.sock\""),
        ("DEBUG", "level=debug msg=\"cgroup memory limit for {cid}: 512m\""),
    ],
}

# ── ANOMALY TEMPLATES ──────────────────────────────────────────────────────────
ANOMALY_TEMPLATES = {

    "user-service": [
        # Exceptions
        ("ERROR", "NullPointerException at {cls}"),
        ("ERROR", "java.sql.SQLTimeoutException: Connection timeout after 30000ms to database {db}"),
        ("ERROR", f"Unhandled exception in request pipeline: {stacktrace(3)}"),
        # Auth attacks
        ("WARN",  "Brute-force detected: 10 consecutive failed logins for user_{uid} from 192.168.1.{ip} — account locked"),
        ("WARN",  "Multiple active sessions detected for user_{uid} from {n} different IPs — possible session hijack"),
        ("ERROR", "JWT signature verification failed token=ey...{hex} expected_iss=devinsight got=unknown"),
        ("WARN",  "Token refresh rejected: original token already revoked user_{uid}"),
        ("ERROR", "Spring Security AccessDeniedException: user_{uid} (ROLE_DEVELOPER) attempted POST /api/admin/users"),
        # DB / pool exhaustion
        ("ERROR", "HikariPool-1 - Connection is not available, request timed out after 30000ms"),
        ("WARN",  "HikariPool-1 - Pool utilisation at 95% (19/20 connections active) — possible leak"),
        ("ERROR", "JDBC connection to {db} refused after 3 retries"),
        # Memory
        ("WARN",  "High heap usage: 88% ({used}MB / {total}MB) — GC overhead limit approaching"),
        ("ERROR", "java.lang.OutOfMemoryError: Java heap space at {cls}"),
        # OAuth
        ("ERROR", "OAuth2 token exchange failed provider=google error=invalid_grant"),
        ("WARN",  "LDAP server ldap://corp-directory:389 unreachable — falling back to local auth"),
    ],

    "payment-service": [
        # Exceptions
        ("ERROR", "java.lang.OutOfMemoryError: Java heap space"),
        ("FATAL", "Transaction {tx} failed: DataIntegrityViolationException — duplicate idempotency key"),
        ("ERROR", f"Critical error in payment pipeline: {stacktrace(4)}"),
        # Gateway failures
        ("ERROR", "Payment gateway {gateway} connection refused after 3 retries: ConnectionRefusedError"),
        ("ERROR", "Payment gateway {gateway} timeout after 10000ms for {tx} — circuit breaker OPEN"),
        ("WARN",  "Circuit breaker [{gateway}] state=OPEN — fast-failing requests for 30s"),
        ("ERROR", "3rd-party gateway {gateway} returned HTTP 502 for {tx}"),
        # Transaction errors
        ("ERROR", "Transaction rollback: {tx} — deadlock detected in DB cluster {db}"),
        ("ERROR", "Decimal precision error: expected {amount} {currency} got {conv_amount} — rounding mismatch"),
        ("WARN",  "Partial payment detected for {tx}: received {amount} expected {total} {currency}"),
        ("FATAL", "Fraud detection ALERT: score=0.94 for {tx} from 192.168.1.{ip} — transaction BLOCKED"),
        ("ERROR", "Rate limit exceeded: merchant_{uid} sent 51/50 requests — returning 429"),
        # Refund
        ("ERROR", "Refund for {tx} failed: original transaction not found in payment-service DB"),
        ("WARN",  "Refund {tx} in PENDING state >10min — possible stuck saga"),
    ],

    "api-gateway": [
        # HTTP errors in access log format
        ("ERROR", "192.168.1.{ip} - - [{date}] \"POST /api/v1/payments HTTP/1.1\" 502 {bytes} \"-\" \"{agent}\""),
        ("ERROR", "192.168.1.{ip} - - [{date}] \"GET /api/v1/users HTTP/1.1\" 503 {bytes} \"-\" \"{agent}\""),
        ("WARN",  "192.168.1.{ip} - - [{date}] \"GET /admin/.env HTTP/1.1\" 403 {bytes} \"-\" \"curl/7.68.0\" [SUSPICIOUS]"),
        ("WARN",  "192.168.1.{ip} - - [{date}] \"GET /../../../etc/passwd HTTP/1.1\" 400 {bytes} \"-\" \"python-requests/2.28.0\" [PATH_TRAVERSAL]"),
        ("WARN",  "192.168.1.{ip} - - [{date}] \"POST /api/v1/login HTTP/1.1\" 429 {bytes} \"-\" \"{agent}\" [RATE_LIMITED]"),
        ("ERROR", "192.168.1.{ip} - - [{date}] \"POST /api/v1/upload HTTP/1.1\" 413 {bytes} \"-\" \"{agent}\" [BODY_TOO_LARGE]"),
        # Gateway internals
        ("ERROR", "upstream timed out (110: Connection timed out) while reading response header from upstream user-service:8081"),
        ("ERROR", "upstream timed out (110: Connection timed out) while reading response header from upstream payment-service:8082"),
        ("ERROR", "no live upstreams while connecting to upstream payment-service [all instances unhealthy]"),
        ("WARN",  "SSL certificate expires in 7 days — renew immediately"),
        ("ERROR", "SSL handshake failure: peer certificate rejected by client"),
        ("WARN",  "CORS violation: origin=https://evil.example.com blocked"),
        ("ERROR", "JWT decode error: malformed token from 192.168.1.{ip} — possible tampering"),
        ("WARN",  "DDoS pattern detected: 192.168.1.{ip} sent {n} requests in 1s — IP throttled"),
        ("ERROR", "Request header 'Content-Type' missing on POST /api/v1/payments from 192.168.1.{ip}"),
    ],

    "k8s-kubelet": [
        # Pod restarts / crashes
        ("WARN",  "Back-off restarting failed container payment-service in pod {ns}/payment-service-{pod} restart_count={n}"),
        ("ERROR", "Pod sandbox changed, it will be killed and re-created: {ns}/user-service-{pod}"),
        ("ERROR", "Container payment-service in pod {ns}/payment-service-{pod} OOMKilled: limit=512Mi used=620Mi"),
        ("ERROR", "Liveness probe FAILED for {ns}/payment-service-{pod}: HTTP probe returned 500 (3/3 failures) — restarting"),
        ("WARN",  "Readiness probe FAILED for {ns}/user-service-{pod}: connection refused on :8081 (1/3 failures)"),
        ("ERROR", "Pod {ns}/payment-service-{pod} exceeded resource quota: requested 4Gi memory, quota allows 2Gi"),
        # Image pull
        ("ERROR", "Failed to pull image \"{img}\": rpc error: code=Unknown desc=context deadline exceeded after 300s"),
        ("WARN",  "ImagePullBackOff: {img} — retry in 5m (attempt 3/5)"),
        # Node pressure
        ("WARN",  "Node {node}: MemoryPressure=True — evicting low-priority pods"),
        ("WARN",  "Node {node}: DiskPressure=True — available=1.2Gi threshold=2Gi"),
        ("ERROR", "Node {node}: NotReady — kubelet stopped posting node status after 40s"),
        ("WARN",  "Node {node}: CPUThrottling detected for container user-service throttled_period=82%"),
        # PV
        ("ERROR", "Failed to mount PersistentVolume {vol}: volume is already exclusively attached to another node"),
        ("ERROR", "PersistentVolumeClaim {ns}/data-pvc stuck in Pending: no matching PV available"),
    ],

    "docker-daemon": [
        # Container failures
        ("ERROR", "level=error msg=\"Handler for POST /v1.41/containers/create returned error: no space left on device\""),
        ("ERROR", "level=error msg=\"failed to start container {cid}: OCI runtime create failed: container process is already dead\""),
        ("ERROR", "level=error msg=\"Container {cid} health check FAILED (3/3) — marking unhealthy\""),
        ("ERROR", "level=error msg=\"Container {cid} exited with code 137 (OOMKill) — restarting per policy\""),
        ("WARN",  "level=warn msg=\"Container {cid} restart count={n} — restart policy on-failure triggered\""),
        # Volume / FS
        ("ERROR", "level=error msg=\"Volume mount failed {vol}: permission denied (uid=1000 gid=1000)\""),
        ("ERROR", "level=error msg=\"Volume {vol} corrupted: fsck returned exit code 4 — manual intervention required\""),
        # Network
        ("ERROR", "level=error msg=\"Network bridge devinsight_default conflict: CIDR 172.18.0.0/16 already in use\""),
        ("ERROR", "level=error msg=\"Container {cid} DNS resolution failed for payment-service.devinsight.svc.cluster.local\""),
        # Registry
        ("ERROR", "level=error msg=\"Registry auth failed for registry.hub.docker.com: unauthorized — check credentials\""),
        ("ERROR", "level=error msg=\"Image layer sha256:{hex}...{hex} corrupted during pull — retrying\""),
        # Daemon
        ("ERROR", "level=error msg=\"cgroup memory limit exceeded for {cid}: usage={n}MB limit={total}MB — process killed\""),
        ("ERROR", "level=error msg=\"Docker daemon lost connection to containerd socket — attempting reconnect\""),
        ("WARN",  "level=warn msg=\"Disk usage at 91% on /var/lib/docker — pruning recommended\""),
    ],
}

# ── Entry generator ────────────────────────────────────────────────────────────
def fill(template: str, ts: datetime) -> str:
    uid   = random.randint(1000, 9999)
    tx    = rtx()
    amount = round(random.uniform(10.0, 5000.0), 2)
    total  = round(amount + random.uniform(1, 50), 2)
    return (
        template
        .replace("{uid}",        str(uid))
        .replace("{tx}",         tx)
        .replace("{amount}",     str(amount))
        .replace("{total}",      str(total))
        .replace("{conv_amount}",str(round(amount * 0.82, 2)))
        .replace("{currency}",   rcur())
        .replace("{gateway}",    rgw())
        .replace("{ip}",         str(rip()))
        .replace("{n}",          str(random.randint(2, 50)))
        .replace("{bytes}",      str(random.randint(120, 8192)))
        .replace("{date}",       ts.strftime("%d/%b/%Y:%H:%M:%S +0000"))
        .replace("{agent}",      random.choice(HTTP_AGENTS))
        .replace("{db}",         random.choice(DB_CLUSTERS))
        .replace("{cls}",        rclass())
        .replace("{ns}",         rns())
        .replace("{node}",       rnode())
        .replace("{pod}",        uuid.uuid4().hex[:6])
        .replace("{img}",        rimg())
        .replace("{vol}",        rvol())
        .replace("{cid}",        uuid.uuid4().hex[:12])
        .replace("{hex}",        uuid.uuid4().hex[:12])
        .replace("{used}",       str(random.randint(600, 900)))
        .replace("{total}",      str(random.randint(1024, 2048)))
    )


def generate_log_entry(current_time: datetime, is_anomaly: bool) -> dict:
    service = random.choice(SERVICES)
    pool    = ANOMALY_TEMPLATES[service] if is_anomaly else NORMAL_TEMPLATES[service]
    level, template = random.choice(pool)
    message = fill(template, current_time)

    return {
        "log_id":          str(uuid.uuid4()),
        "timestamp":       current_time.isoformat() + "Z",
        "level":           level,
        "service_name":    service,
        "message":         message,
        "is_true_anomaly": is_anomaly,
    }


# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    print(f"Generating {NUM_LOGS} synthetic logs ({int(ANOMALY_RATIO*100)}% anomalies)…")

    logs           = []
    current_time   = datetime.utcnow() - timedelta(hours=1)
    normal_count   = 0
    anomaly_count  = 0

    for _ in range(NUM_LOGS):
        current_time += timedelta(milliseconds=random.randint(10, 2000))
        is_anomaly    = random.random() < ANOMALY_RATIO

        if is_anomaly:
            anomaly_count += 1
        else:
            normal_count += 1

        logs.append(generate_log_entry(current_time, is_anomaly))

    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, "w") as f:
        json.dump(logs, f, indent=2)

    print(f"\nDone!")
    print(f"  Normal logs  : {normal_count}")
    print(f"  Anomaly logs : {anomaly_count}")
    print(f"  Total        : {len(logs)}")
    print(f"  Output       : {OUTPUT_FILE}")

    # ── Coverage summary ───────────────────────────────────────────────────────
    from collections import Counter
    by_service = Counter(l["service_name"] for l in logs)
    by_level   = Counter(l["level"]        for l in logs)
    print("\nBy service:")
    for svc, n in sorted(by_service.items()):
        print(f"  {svc:<25} {n}")
    print("\nBy level:")
    for lvl, n in sorted(by_level.items()):
        print(f"  {lvl:<10} {n}")


if __name__ == "__main__":
    main()

Generating 10000 synthetic logs (5% anomalies)…

Done!
  Normal logs  : 9490
  Anomaly logs : 510
  Total        : 10000
  Output       : data/synthetic_logs.json

By service:
  api-gateway               2005
  docker-daemon             1972
  k8s-kubelet               2037
  payment-service           1987
  user-service              1999

By level:
  DEBUG      1637
  ERROR      321
  FATAL      13
  INFO       7853
  WARN       176
